# Phase 3 - Severity head  —  Rotax_915_iSHow bad is each channel, 0..1. Takes the encoder output **plus** the diagnosisprobabilities - knowing *which* fault it is informs *how severe*.

In [ ]:
import os, sys, json, timesys.path.insert(0, os.path.abspath("."))import numpy as np, tensorflow as tffrom tensorflow import kerasimport tf_data_pipeline as P, model_architectures as A, train_common as CC.report_gpu()

In [ ]:
# ---- SET THE ENGINE HERE. Run this notebook once per engine. ----ENGINE = "Rotax_915_iS"steps = C.describe(ENGINE)train_ds, val_ds, test_ds = C.datasets(ENGINE)

### MetricWatch `nonzero_severity_mse`, not plain MSE: zeros dominate, so plain MSE looksexcellent while the head predicts nothing at all.

In [ ]:
xi  = A.build_encoder_input()enc = A.build_encoder(xi)det = A.build_detection_head(enc)dia = A.build_diagnosis_head(enc)sev = A.build_severity_head(enc, dia)model = keras.Model(xi, {"y_detection": det, "y_diagnosis": dia, "y_severity": sev},                    name="severity")model.summary()

In [ ]:
cw = np.load(os.path.join(C.MODELS_DIR, f"diag_class_weights_{C.ENGINE_KEY[ENGINE]}.npy"))print("class weights:", cw.round(3))

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),              loss={"y_detection": keras.losses.BinaryCrossentropy(),                    "y_diagnosis": C.weighted_diagnosis_loss(cw),                    "y_severity": keras.losses.MeanSquaredError()},              loss_weights={"y_detection": 1.0, "y_diagnosis": 50.0, "y_severity": 20.0},              metrics={"y_severity": [C.nonzero_severity_mse, C.nonzero_severity_mae]})

In [ ]:
PHASE = "phase3_severity"PREV  = "phase2_diagnosis"ckpt  = C.ckpt_path(ENGINE, PHASE)if PREV: C.warm_start(model, C.ckpt_path(ENGINE, PREV))hist = model.fit(    train_ds.map(C.split_labels(["y_detection","y_diagnosis","y_severity"])),    validation_data=val_ds.map(C.split_labels(["y_detection","y_diagnosis","y_severity"])),    steps_per_epoch=steps["train"],    validation_steps=steps["val"],    epochs=20,    callbacks=C.callbacks(ckpt, monitor="val_y_severity_nonzero_severity_mse", mode="min"),    verbose=1,)print("saved:", ckpt)

In [ ]:
res = model.evaluate(test_ds.map(C.split_labels(["y_detection","y_diagnosis","y_severity"])), steps=steps["test"], verbose=1, return_dict=True)print(json.dumps({k: float(v) for k, v in res.items()}, indent=2))with open(os.path.join(C.MODELS_DIR, f"{PHASE}_{C.ENGINE_KEY[ENGINE]}_test.json"), "w") as f:    json.dump({k: float(v) for k, v in res.items()}, f, indent=2)